# 06 · Decoder & submission

**Tujuan.** Menggabungkan keputusan bola hidup/mati (`03`) dan bukti tim (`05`) menjadi satu label per frame,
menulis file submission, memvalidasinya dengan validator resmi, dan mengukurnya dengan **proxy tanpa GT**.

| | |
|---|---|
| **Input** | `cache/dead.npz`, `cache/restarts.csv`, `cache/possession_model.npz`, `cache/tags.csv` |
| **Output** | `submissions/<tim>_<versi>.csv`, laporan `cache/reports/06_decoder_submission.*` |
| **Waktu** | ± 1 menit |

### Riwayat versi

| Versi | `D` | Bukti H/A | Catatan |
|---|---|---|---|
| v0 | `03` v3 | tim terdekat ke bola (±1 per frame), `SWITCH_HA` = 25 | acuan Fase 1 (`docs/experiment_log.md` #1) |
| v1 | `03` v3 | log-odds `05`: bentuk tim (out-of-fold) + β × tim terdekat | Fase 2.6 (#2) |
| **v2** | **`03` v4** (jangkar + aktivitas operan) | sama dengan v1 | Fase 3.1 — notebook ini |

### Decoder v1

Viterbi 3 keadaan (0 = `H`, 1 = `A`, 2 = `D`) per babak:

| Keadaan | Emisi di frame hidup | Emisi di frame `D` (`03`) |
|---|---|---|
| `H` | +½ · clip(`L_total`, ±`L_CLIP`) | −10⁴ |
| `A` | −½ · clip(`L_total`, ±`L_CLIP`) | −10⁴ |
| `D` | −10⁴ | +10⁴ |

Selisih emisi H − A per frame = log-odds. Mengganti H↔A membayar **`SWITCH_HA` = `SWITCH_S` × 25 × median |L|**:
bukti untuk tim baru harus setara ± `SWITCH_S` detik bukti "biasa". Biaya dinyatakan dalam **detik bukti**, sehingga
tidak bergantung pada skala log-odds model — dan tidak disetel di GT publik. Masuk/keluar `D` gratis karena `D` sudah
diputuskan `03`.

`clip` mencegah satu frame dengan log-odds ekstrem (mis. tracking rusak) mendominasi.

In [ ]:
%run ./common.ipynb

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `VERSION` | `v2` | dicatat di nama file dan log eksperimen |
| `SWITCH_S` | 1,0 s | pergantian possession butuh kontrol yang bertahan (CHALLENGE.md); 1 s = skala `CTRL_MIN` ×2. Tidak disetel di GT publik — lihat tabel sensitivitas di bagian 4 |
| `L_CLIP` | 6 | batas log-odds per frame (P ≈ 0,998) |
| `BOOTSTRAP_N` | 1000 | interval bootstrap resmi untuk skor publik |

In [ ]:
VERSION = "v2"
SWITCH_S = 1.0
L_CLIP = 6.0
BOOTSTRAP_N = 1000
SUBMISSION_NAME = "{}_{}".format(TEAM_NAME, VERSION)

In [ ]:
rep = Report("06_decoder_submission")
with np.load(CACHE / "dead.npz") as dz:
    d_mask = dz["d_mask"]
with np.load(CACHE / "possession_model.npz") as mz:
    L_total, pseudo = mz["L_total"].astype(np.float64), mz["pseudo"]
restarts = pd.read_csv(CACHE / "restarts.csv", keep_default_na=False)
tags = load_tags()
live = GRADED & ~d_mask
L = np.clip(L_total, -L_CLIP, L_CLIP)
L_MED = float(np.median(np.abs(L[live])))
rep("versi {} | porsi D {:.3f} | median |L| di frame hidup {:.3f}".format(VERSION, d_mask[GRADED].mean(), L_MED))

## 1. Decoder

In [ ]:
STATES = np.array(["H", "A", "D"])
BIG = 1e4
emis = np.zeros((N_FRAMES, 3))
emis[:, 0], emis[:, 1] = 0.5 * L, -0.5 * L
emis[d_mask, 0] = emis[d_mask, 1] = -BIG
emis[d_mask, 2] = BIG
emis[~d_mask, 2] = -BIG


def decode(switch_s):
    cost = switch_s * FPS * L_MED
    trans = np.array([[0.0, cost, 0.0], [cost, 0.0, 0.0], [0.0, 0.0, 0.0]])
    out = np.full(N_FRAMES, "X", dtype="<U1")
    for h, (a, b) in HALVES.items():
        out[a:b + 1] = STATES[viterbi(emis[a:b + 1], trans)]
    return out


with timer("viterbi"):
    labels = decode(SWITCH_S)
rep("SWITCH_HA = {:.1f} (= {} s × {} × {:.3f})".format(SWITCH_S * FPS * L_MED, SWITCH_S, FPS, L_MED))

## 2. Proxy tanpa GT

Dihitung di seluruh match, tanpa label apa pun dari GT:

| Proxy | Arti | Catatan |
|---|---|---|
| `possession`, `durasi rata-rata` | jumlah & durasi possession yang dilaporkan | kewajaran; GT publik: 17 possession, 14,2 s (5 menit pertama saja) |
| `< 2 s %` | porsi possession sangat pendek | terlalu banyak = berkedip; nol = tak menangkap possession pendek (GT punya beberapa) |
| `pseudo-label cocok %` | label decoder = pseudo-label "kontrol jelas" (`05`) | sebagian melingkar (pemain terdekat ikut dalam bukti) — batas atas |
| `restart cocok %` | label 0,5 s setelah setiap `D` = tim restart terdekat (`03`) | konsistensi dengan temuan `02` #9 |
| `porsi H hidup` | porsi H di waktu hidup | kewajaran (Barcelona di kandang) |

In [ ]:
def proxies(lab):
    g = lab[GRADED]
    spells = segments_of(lab[GRADED], 0)
    spells = spells[spells.label != "D"]
    # possession tidak melintasi jeda turun minum: segments_of di frame dinilai yang digabung bisa menyambung
    # akhir babak 1 dengan awal babak 2, tetapi awal babak 2 selalu D (prior), jadi tidak ada sambungan H/A.
    pl = live & (pseudo != 0)
    pl_agree = float((np.where(lab[pl] == "H", 1, -1) == pseudo[pl]).mean())
    rs = restarts[restarts.restart_team.isin(["H", "A"])]
    idx = np.clip(rs.restart_frame.to_numpy() + 12, 0, N_FRAMES - 1)
    rs_agree = float((lab[idx] == rs.restart_team.to_numpy()).mean())
    alive = np.isin(g, ["H", "A"])
    return {"possession": len(spells), "durasi rata-rata (s)": round(float(spells.dur_s.mean()), 1),
            "< 2 s %": round(100 * float((spells.dur_s < 2).mean()), 1),
            "pseudo-label cocok %": round(100 * pl_agree, 1), "restart cocok %": round(100 * rs_agree, 1),
            "porsi H hidup": round(float((g[alive] == "H").mean()), 3)}


prox = proxies(labels)
rep("proxy v1: {}".format(prox))

## 3. Tulis & validasi

In [ ]:
path, digest = write_submission(labels, SUBMISSION_NAME)
ok, summary = validate_file(path)
rep("file: {} | sha256 {}".format(path.relative_to(ROOT), digest))
rep("validasi resmi: {} | {}".format("LULUS" if ok else "GAGAL", summary))
assert ok, summary

## 4. Sensitivitas terhadap `SWITCH_S`

Untuk memilih **bukan** dengan GT publik, tabel ini memperlihatkan bagaimana proxy berubah. Kolom publik
disertakan sebagai informasi (tanpa bootstrap, jadi bacalah sebagai kasar; satu possession ≈ ±6 poin).
Nilai yang baik berada di **dataran** — proxy tidak berubah tajam di sekitarnya.

In [ ]:
sens = []
tmp = CACHE / "_tmp_sens"
tmp.mkdir(exist_ok=True)
with timer("sensitivitas"):
    for s in (0.25, 0.5, 1.0, 1.5, 2.0, 3.0):
        lab_s = decode(s)
        row = {"SWITCH_S": s, **proxies(lab_s)}
        p_s, _ = write_submission(lab_s, "sens", folder=tmp)
        sc = score_public(p_s)
        row["publik spellF1"] = round(sc["spell_f1_mean"], 3)
        row["publik macroF1"] = round(sc["macro_f1"], 3)
        sens.append(row)
        p_s.unlink()
tmp.rmdir()
sens = pd.DataFrame(sens).set_index("SWITCH_S")
rep.table(sens, "sensitivitas SWITCH_S:")

## 5. Cek kebenaran pada GT publik (dengan interval bootstrap)

In [ ]:
cols = ["spell_f1_mean", "spell_bootstrap_low", "spell_bootstrap_high", "macro_f1", "n_pred_spells",
        "n_gt_spells", "mean_spell_duration_s", "alive_dead_f1", "team_acc_gt_alive"]
board = {}
with timer("skor publik + bootstrap"):
    board[SUBMISSION_NAME] = score_public(path, bootstrap_n=BOOTSTRAP_N)
    for old in ("v1", "v0"):
        prev = SUBMISSIONS / "{}_{}.csv".format(TEAM_NAME, old)
        if prev.is_file():
            board[prev.stem] = score_public(prev, bootstrap_n=BOOTSTRAP_N)
    board["baseline_smooth50 (resmi)"] = score_public(
        OFFICIAL / "evaluation" / "results" / "baseline_smooth50.csv", bootstrap_n=BOOTSTRAP_N)
board_df = pd.DataFrame(board).T[cols].astype(float).round(4)
rep.table(board_df, "GT publik:")
rep.save({"version": VERSION, "file": str(path.relative_to(ROOT)), "sha256": digest, "summary": summary,
          "proxies": prox, "sensitivity": sens.reset_index().to_dict("records"),
          "public": board_df.reset_index().to_dict("records"),
          "params": {"SWITCH_S": SWITCH_S, "L_CLIP": L_CLIP, "L_MED": L_MED}})

## Ringkasan & keputusan (v2)

| GT publik | **v2** | v1 | v0 | baseline resmi |
|---|---|---|---|---|
| spellF1 [bootstrap 5–95%] | **± 0,59 [0,37 – 0,77]** | ± 0,54 [0,32 – 0,74] | ± 0,34 [0,19 – 0,50] | ± 0,33 [0,21 – 0,45] |
| macroF1 | **± 0,86** | ± 0,77 | ± 0,72 | ± 0,62 |
| F1 hidup/mati | ± 0,95 | ± 0,88 | ± 0,88 | ± 0,90 |
| akurasi tim pada frame hidup GT | ± 0,89 | ± 0,75 | ± 0,69 | ± 0,83 |
| possession (GT 17) | 11 | 12 | 19 | 26 |

Seluruh match (proxy tanpa GT): ± 186 possession, rata-rata ± 18 s; pseudo-label cocok ± 92%; restart cocok ± 79%.

**Bacaan**

1. **`D` v4 mengembalikan frame hidup yang dicuri `D` palsu:** akurasi tim pada frame hidup naik dari ± 0,75 ke ± 0,89
   (mendekati ± 0,92 milik model bentuk tim), macroF1 naik ± 0,09.
2. **spellF1 naik lebih sedikit** karena pembatasnya kini lain: **terlalu sedikit possession** (11 vs 17). Possession
   pendek dan pergantian cepat tidak tertangkap — decoder (dan bentuk tim yang berubah lambat) enggan berganti.
3. `SWITCH_S` tetap tidak rapuh: skor publik identik untuk 0,25–1,5 s.

**Keputusan:** v2 menjadi acuan baru (`docs/experiment_log.md` #3). Berikutnya **2.7 — pergantian & batas**:
bukti bola (kontrol oleh tim baru) untuk menangkap possession pendek dan menaruh batas di saat yang tepat, tanpa
membuat possession palsu.